In [1]:
import mne
import glob 
import numpy as np
%matplotlib qt

In [ ]:
# EEG Processing Pipeline
# This pipeline is designed for resting-state EEG analysis, specifically for extracting aperiodic and periodic components in further analysis.
# Record of Revisions, 
#
# Date            Programmers                         Descriptions of Change
# ===========  ======================   ============================================
# 12-May-2021  Amin Ghaderi Kangavari   Created initial task-based EEG preprocessing pipeline in MNE
# 22-Nov-2024  Amin Ghaderi Kangavari   Customized for rs-EEG, added CSD, shifted  re-referencing, removed all non-EEG channels, used for a-periodic components
# 06-Feb-2026  Amin Ghaderi Kangavari   Add some properties to distinguish better ICs, automatic selection of n_components in ICA
# 16-Apr-2026  Amin Ghaderi Kangavari   Added manual bad segment annotation via Qt raw plot, cleaned script

In [ ]:
# Main directory (set your folder)
main_dir = '/eeg/Derivatives/EEG_rest'
parts = []
# Find EEG files
for file in main_dir.glob("*/*_rest.bdf"):
    # Extract participant name (parent folder)
    sub_name = file.parent.name
    parts.append(sub_name)
parts.sort()

#### Load EEG data

In [ ]:
# Select a participant
subject = parts[20]
print(f'{subject} is preprocessing')
# Read EEG raw file (BDF format)
filename = f"{main_dir}/{subject}/{subject}_rest.bdf"
raw = mne.io.read_raw_bdf(filename, preload=True)
# Set BioSemi 128 montage
raw.set_montage('biosemi128', on_missing="ignore")
#raw.plot_sensors(show_names=True)
# Drop all non-EEG channels safely
EEG_channels = raw.info['ch_names'][128:]
raw.drop_channels(EEG_channels)

### downsampling

In [ ]:
# Downsample to 1000 per sec, one data point per millisecond
raw.resample(1000, npad="auto")     

### filtering

In [ ]:
# Band-pass filter (0.1–95 Hz)
raw.filter(.1, 95,  h_trans_bandwidth = 4, fir_design='firwin')

# Notch filter for line noise (50 Hz and harmonic at 100 Hz)
freqs = (50, 100)
raw = raw.notch_filter(freqs=freqs, picks='eeg', trans_bandwidth=2)

- visual inspection

In [ ]:
# Any annotation starting with "bad" (e.g. "bad_artifact") is automatically treated as bad data
raw.plot(scalings=dict(eeg=50e-6))

In [ ]:
# Print annotations 
for annot in raw.annotations:        
    print(annot)

In [ ]:
# Plot PSD with weltch method
fmin = .1
fmax = 90
sample_rate = int(raw.info['sfreq']) 
# two second * sampling rate
wind =  2 * sample_rate
# 1 second * sampling rate
overlap =  1 * sample_rate
raw.compute_psd(fmin=fmin, fmax=fmax, method='welch', picks='eeg', n_overlap= overlap,  n_fft=wind).plot()
# Whether to omit bad segments from the data before fitting, reject_by_annotation, is True by default in both power spectrum and ICA

In [ ]:
# Interpolate bad channels
raw.info['bads'] = []
raw = raw.interpolate_bads(reset_bads=False)
# Reset all bad cahnnels
raw.info['bads'] = []

### re-referencing

In [ ]:
# re-referencing with the virtual average reference
raw.set_eeg_reference('average', projection=True).apply_proj()

### Run ICA

In [ ]:
# applying surface laplacian computation, volume conduction 
raw = mne.preprocessing.compute_current_source_density(raw, lambda2=1e-3, stiffness = 4, n_legendre_terms = 80) 

In [ ]:
# Run ICA with fastica method
ica = mne.preprocessing.ICA(random_state=123, method='fastica')
ica.fit(raw, reject_by_annotation=True)                           

In [ ]:
# Automatic muscle-artifact components
muscle_idx_auto, scores = ica.find_bads_muscle(raw)
ica.plot_scores(scores, exclude=muscle_idx_auto) 

In [ ]:
# Plot spatial ICs
ica.plot_components()

In [ ]:
# Plot the timecourse of spatial ICs
ica.plot_sources(raw)

In [ ]:
# Plot all ICs properties
ica.plot_properties(raw, picks=np.linspace(0,ica.n_components_-1,ica.n_components_).astype(int),log_scale=False)

In [ ]:
# plot some ICs properties in log-log scale
ica.plot_properties(raw, picks=[18],log_scale=True)

In [ ]:
# Diagnosis to validate component rejections
ica.plot_overlay(raw, exclude=[1,23,25,26,32], picks="csd")

In [ ]:
# Set list of excluded channels
ica.exclude= [1,23,25,26,32]
# Reset bad cahnnels
raw.info['bads'] = []

In [ ]:
# List of excluded ICs
ica.exclude

In [ ]:
# Plot raw data before applying ICA
raw.plot()

In [ ]:
# Plot PSD in current sourse density domain before applying ICA
fmin = .1
fmax = 90
wind =  2 * int(raw.info['sfreq']) # two second * sampling rate
overlap =  1 * int(raw.info[ 'sfreq']) # 1 second * sampling rate
raw.compute_psd(fmin=fmin, fmax=fmax, method='welch', picks='csd', n_overlap= overlap,  n_fft=wind,
    reject_by_annotation=True).plot()

In [ ]:
# Apply ICA to reconestract clean data by removeing the noisy components
ica.apply(raw)                  

### save preprocessed data

In [ ]:
# Save final preprocessed raw data
save_filename = filename.split('.')[0]
#raw.save(f'{save_filename}_preprocessed.fif', overwrite=True)
print(f'{subject}_preprocessed.fif')

In [ ]:
# Plot raw data after applying ICA
raw.plot()

In [ ]:
# Plot PSD in current sourse density domain after applying ICA
fmin = .1
fmax = 90
# two second * sampling rate
wind =  2 * int(raw.info['sfreq'])
# 1 second * sampling rate
overlap =  1 * int(raw.info[ 'sfreq']) 
raw.compute_psd(fmin=fmin, fmax=fmax, method='welch', picks='csd', n_overlap= overlap,  n_fft=wind).plot()